# Naive RAG - End-to-End Demo

**Pipeline:** PDF -> Chunk -> Embed -> FAISS -> Similarity Search -> LLM Answer

**Stack:** Groq (Llama-3.1-8B-instant) - LangChain - FAISS - sentence-transformers - PyMuPDF

---

## Runs anywhere
Google Colab, Jupyter, JupyterLab, VS Code, Kaggle, local Python, Docker. Setup cells auto-detect your environment.

## Prerequisites

- **Python** 3.9 or newer
- **RAM** 4 GB+
- **Disk** ~500 MB (Hugging Face cache)
- **GROQ_API_KEY** - free at [console.groq.com/keys](https://console.groq.com/keys)
- **Internet** on first run (downloads the model + sample PDF)

## Quick start
1. Run the install cell.
2. Run the setup cell - it will load `GROQ_API_KEY` from env, Colab secrets, `.env`, or prompt.
3. Run the rest top-to-bottom. The demo PDF (*Attention Is All You Need*) is auto-downloaded if missing.

> For the more sophisticated variant (Hybrid retrieval + RRF + Cross-Encoder reranking + RAGAS evaluation) see the **AdvancedRAG** project.

In [ ]:
# ── Install dependencies (works on Colab / Jupyter / VS Code / local) ──
# Note: On Google Colab you may see a harmless warning about google-colab
# requiring requests==2.32.4 — that's a notice, not a failure.
!pip install -q \
    groq gradio python-dotenv \
    langchain langchain-core langchain-community langchain-groq \
    langchain-text-splitters langchain-huggingface \
    sentence-transformers faiss-cpu rank_bm25 pymupdf \
    ragas datasets pandas matplotlib \
    "requests>=2.32.4,<2.33"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 280.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.7/360

In [ ]:
# Importing libraries
import gradio as gr
import os
import re
import numpy as np
from typing import List

# LangChain
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

# Groq
from groq import Groq


In [ ]:
# ══════════════════════════════════════════════════════════════════
# NAIVE RAG — Complete End-to-End
# Flow: PDF → Load → Chunk → Embed → Vector DB → Query → Retrieve → LLM → Answer
# ══════════════════════════════════════════════════════════════════
import os
from getpass import getpass

# ── Portable API key loader — works in Colab, Jupyter, VS Code, local Python ──
def load_groq_key():
    # 1. Already in environment? (CI/CD, Docker, exported in shell)
    if os.environ.get("GROQ_API_KEY"):
        return "environment"
    # 2. Google Colab secrets
    try:
        from google.colab import userdata
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
        return "colab-secrets"
    except (ImportError, ModuleNotFoundError):
        pass
    # 3. .env file (local dev with python-dotenv)
    try:
        from dotenv import load_dotenv
        if load_dotenv() and os.environ.get("GROQ_API_KEY"):
            return "dotenv"
    except ImportError:
        pass
    # 4. Interactive prompt (Jupyter / VS Code / terminal)
    os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ_API_KEY: ").strip()
    return "prompt"

source = load_groq_key()
print(f"✅ GROQ_API_KEY loaded from: {source}")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 0 — Setup LLM client
# ─────────────────────────────────────────────────────────────────
client = Groq(api_key=os.environ["GROQ_API_KEY"])

def call_llm(prompt: str) -> str:
    res = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    return res.choices[0].message.content


In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 1 — READ PDF (auto-downloads if missing — works on any OS)
# ─────────────────────────────────────────────────────────────────
import os, urllib.request, tempfile

# Use /content if on Colab, else use OS-appropriate temp dir
if os.path.isdir("/content"):
    PDF_DIR = "/content"
else:
    PDF_DIR = os.path.join(tempfile.gettempdir(), "rag_demo")
    os.makedirs(PDF_DIR, exist_ok=True)

PDF_PATH = os.path.join(PDF_DIR, "Attention is all you need.pdf")
if not os.path.exists(PDF_PATH):
    print(f"Downloading PDF to {PDF_PATH} ...")
    urllib.request.urlretrieve("https://arxiv.org/pdf/1706.03762.pdf", PDF_PATH)

loader = PyMuPDFLoader(PDF_PATH)
pages = loader.load()
print(f" Step 1 — Loaded {len(pages)} pages from PDF")


 Step 1 — Loaded 15 pages from PDF


In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 2 — CHUNK
# ─────────────────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
docs = splitter.split_documents(pages)

# Add metadata for citations
for i, d in enumerate(docs):
    d.metadata["chunk_id"] = i
    d.metadata["page"] = d.metadata.get("page", 0) + 1

print(f" Step 2 — Split into {len(docs)} chunks")



 Step 2 — Split into 52 chunks


In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 3 — EMBED + STORE in Vector DB (FAISS)
# ─────────────────────────────────────────────────────────────────
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore = FAISS.from_documents(docs, embeddings)
print(f" Step 3 — Stored {len(docs)} chunks in FAISS vector DB")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Step 3 — Stored 52 chunks in FAISS vector DB


In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 4 — NAIVE RAG (retrieve from vector DB → LLM → answer)
# ─────────────────────────────────────────────────────────────────
NAIVE_PROMPT = """You are a precise assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say: "Not found in document".
Cite pages like [Page X].

Context:
{context}

Question: {question}

Answer:"""

def naive_rag(query: str, k: int = 5) -> dict:
    # 4a. RETRIEVE top-k similar chunks from vector DB
    retrieved = vectorstore.similarity_search(query, k=k)

    # 4b. BUILD context string (with page numbers for citations)
    context_blocks = []
    for d in retrieved:
        page = d.metadata.get("page", "?")
        context_blocks.append(f"[Page {page}]\n{d.page_content}")
    context = "\n\n---\n\n".join(context_blocks)

    # 4c. ASK the LLM
    prompt = NAIVE_PROMPT.format(context=context, question=query)
    answer = call_llm(prompt)

    return {
        "answer": answer,
        "retrieved_chunks": retrieved,
        "pages_used": sorted({d.metadata.get("page", "?") for d in retrieved}),
    }


In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 5 — DEMO
# ─────────────────────────────────────────────────────────────────
queries = [
    "What is the main idea of this paper?",
    "What is the Transformer architecture?",
    "What datasets were used in the experiments?",
    "Who are the authors of this paper?",
]

for q in queries:
    print("\n" + "═" * 70)
    print(f" Question: {q}")
    print("═" * 70)
    out = naive_rag(q)
    print(f" Pages used: {out['pages_used']}")
    print(f"\n Answer:\n{out['answer']}")


══════════════════════════════════════════════════════════════════════
 Question: What is the main idea of this paper?
══════════════════════════════════════════════════════════════════════
 Pages used: [1, 12, 13, 14, 15]

 Answer:
Not found in document.

══════════════════════════════════════════════════════════════════════
 Question: What is the Transformer architecture?
══════════════════════════════════════════════════════════════════════
 Pages used: [1, 2, 3, 5, 8]

 Answer:
The Transformer architecture follows an overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, as shown in Figure 1 [Page 3].

══════════════════════════════════════════════════════════════════════
 Question: What datasets were used in the experiments?
══════════════════════════════════════════════════════════════════════
 Pages used: [4, 7, 9]

 Answer:
Not found in document.

═══════════════════════════════════════════════════════════════